In [1]:
import pandas as pd
import re
from dateutil import parser
import warnings
warnings.filterwarnings('ignore', message='This pattern is interpreted as a regular expression')
warnings.filterwarnings("ignore")

# import everything 
from clean_data import(load_and_clean_books_data, english_only, normalize_text, normalize_title, remove_non_english_titles, 
remove_duplicate_book_ids, create_book_identifier, clean_description, create_is_boxset_flag, normalize_pages)
from clean_author import normalize_author_text, normalize_author, normalize_author_roles, create_author_role_columns, filter_invalid_authors
from clean_rating import normalize_num_ratings, normalize_ratings_by_stars
from clean_genre import normalize_genre, clean_genres_column    
from clean_publisher import normalize_publishers, remove_non_english_publishers
from clean_isbn import create_isbn_cols, remove_fake_isbn13, keep_one_per_isbn13, keep_one_per_isbn10, remove_rows_without_isbn
from clean_series import parse_series, is_part_of_series, create_series_identifier
from clean_bookformat import  normalize_book_format
from clean_edition import normalize_edition
from clean_publisher import normalize_publishers
from clean_publishdate import normalize_publish_date
from run import run_cleaning

In [2]:
# run the cleaning pipeline
books_data = load_and_clean_books_data("data/books.csv")
print(f"Original data shape: {books_data.shape}")
cleaned_books_data = run_cleaning(books_data)
print(f"Cleaned data shape: {cleaned_books_data.shape}")

Original data shape: (52478, 18)
Cleaned data shape: (35362, 67)


In [3]:
# count non "English" language 
print("Language distribution before cleaning:")
print(books_data['language'].value_counts())

print("\nLanguage distribution after cleaning:")
print(cleaned_books_data['language'].value_counts())

Language distribution before cleaning:
language
English                                  42661
Arabic                                    1038
Spanish                                    687
French                                     579
German                                     528
                                         ...  
Aromanian; Arumanian; Macedo-Romanian        1
Basque                                       1
Faroese                                      1
Iranian (Other)                              1
Norwegian Nynorsk; Nynorsk, Norwegian        1
Name: count, Length: 81, dtype: int64

Language distribution after cleaning:
language
English    35362
Name: count, dtype: int64


In [4]:
# title vs title_norm (pd dataframe with columns title and title_norm, show 5 rows)
print("\nTitle vs Title_norm:")
print(cleaned_books_data[['title', 'title_norm']].head().to_string())


Title vs Title_norm:
                                                                       title                                                                 title_norm
0                                                        Think and Grow Rich                                                        think and grow rich
1                                              The Message for the Last Days                                              the message for the last days
2                                                      Reckless Endangerment                                                      reckless endangerment
3  Easy Money Management: A Give and Save 3-6-5 Approach to Personal Finance  easy money management: a give and save 3-6-5 approach to personal finance
4                                                      The Selfish Betrayals                                                      the selfish betrayals


In [5]:
# author column
print("\nAuthor vs Author_norm:")
print(cleaned_books_data[['author', 'author_norm']].head(5).to_string())

# author vs author columns (primary_author, author_2...)
print("\nAuthor vs Seperated Author columns:") # not author_roles_norm
author_cols = ['primary_author', 'author_2', 'author_3', 'author_4', 'author_5']
print(cleaned_books_data[['author'] + author_cols].head(5).to_string())


Author vs Author_norm:
                                                  author                         author_norm
0  napoleon hill, ben holden-crowther (goodreads author)  napoleon hill, ben holden-crowther
1                             kj soze (goodreads author)                             kj soze
2                    amber lea easton (goodreads author)                    amber lea easton
3                      laurick ingram (goodreads author)                      laurick ingram
4                     abhishek kapoor (goodreads author)                     abhishek kapoor

Author vs Seperated Author columns:
                                                  author    primary_author             author_2 author_3 author_4 author_5
0  napoleon hill, ben holden-crowther (goodreads author)     napoleon hill  ben holden-crowther     None     None     None
1                             kj soze (goodreads author)           kj soze                 None     None     None     None
2           

In [6]:
# publish_date vs publishDate_norm 
print("\nPublish Date vs Publish Date Norm:")
print(cleaned_books_data[['publishDate', 'publishDate_norm']].head(5).to_string())


Publish Date vs Publish Date Norm:
          publishDate publishDate_norm
0         April 2016        2016-04-22
1      June 17th 2019       2019-06-17
2     April 12th 2013       2013-04-12
3  November 30th 2018       2018-11-30
4            May 2019       2019-05-22


In [7]:
# get book_uid with more than 1 row (duplicate book_uid)
duplicate_book_uids = cleaned_books_data['book_uid'].value_counts()
duplicate_book_uids = duplicate_book_uids[duplicate_book_uids > 1]
print("\nDuplicate book_uids:")
print(duplicate_book_uids)

# example for book_uid 80
print("\nExample of duplicate book_uid (80):")
print(cleaned_books_data[cleaned_books_data['book_uid'] == 311].to_string())


Duplicate book_uids:
book_uid
80       3
1011     2
16120    2
311      2
17494    2
269      2
28103    2
4460     2
864      2
235      2
67       2
792      2
5791     2
466      2
6036     2
1357     2
673      2
22001    2
385      2
3871     2
303      2
919      2
220      2
Name: count, dtype: int64

Example of duplicate book_uid (80):
               bookId     title series                                                       author  rating                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [8]:
# get all books where 'george rr martin in the author_norm column
print("\nBooks where 'george rr martin' is in the author_norm column:")
# only columns: title, titme_norm, author, author_norm, genre, genre_norm, publishDate, publishDate_norm
print(cleaned_books_data[cleaned_books_data['author_norm'].str.contains('george rr martin', na=False)]
      [['title', 'title_norm', 'primary_author', 'genres_norm', 'publishDate', 'publishDate_norm']].head(5).to_string())

# no do the same but get from uncleaned data
print("\nBooks where 'george rr martin' is in the author column (uncleaned data):")
print(books_data[books_data['author'].str.contains('George R.R. Martin', na=False)]
      [['title', 'author', 'genres', 'publishDate']].head(5).to_string())


Books where 'george rr martin' is in the author_norm column:
                                  title                         title_norm    primary_author                                                                                    genres_norm         publishDate publishDate_norm
49                    A Game of Thrones                  a game of thrones  george rr martin  [fantasy, fiction, epic fantasy, adult, science fiction, adventure, dragons, audiobook, epic]            08/28/05       2005-08-28
159                   A Storm of Swords                  a storm of swords  george rr martin  [fantasy, fiction, epic fantasy, science fiction, adult, adventure, dragons, audiobook, epic]            03/04/03       2003-03-04
199                    A Clash of Kings                   a clash of kings  george rr martin  [fantasy, fiction, epic fantasy, science fiction, adult, adventure, dragons, audiobook, epic]            05/28/02       2002-05-28
338                   A Feast for Crow

In [9]:
# now same for author of hunger games, suzanne collins
print("\nBooks where 'suzanne collins' is in the author_norm column:")
print(cleaned_books_data[cleaned_books_data['author_norm'].str.contains('suzanne collins', na=False)]
      [['title', 'title_norm', 'primary_author', 'genres_norm', 'publishDate', 'publishDate_norm']].head(5).to_string()) 

print("\nBooks where 'suzanne collins' is in the author column (uncleaned data):")
print(books_data[books_data['author'].str.contains('Suzanne Collins', na=False)]
      [['title', 'author', 'genres', 'publishDate']].head(5).to_string())   


Books where 'suzanne collins' is in the author_norm column:
                                title                       title_norm   primary_author                                                                                                      genres_norm   publishDate publishDate_norm
19                   The Hunger Games                 the hunger games  suzanne collins  [young adult, fiction, dystopian, fantasy, science fiction, romance, adventure, teen, post-apocalyptic, action]      09/14/08       2008-09-14
152   The Hunger Games Trilogy Boxset  the hunger games trilogy boxset  suzanne collins                    [young adult, fiction, fantasy, dystopian, science fiction, romance, adventure, teen, action]      08/24/10       2010-08-24
178                     Catching Fire                    catching fire  suzanne collins  [young adult, dystopian, fiction, fantasy, science fiction, romance, adventure, teen, post-apocalyptic, action]      09/01/09       2009-09-01
256        

In [10]:
# top 10 rows of author column in uncleaned data
print("\nTop 10 rows of author column in uncleaned data:")
print(books_data['author'].head(10).to_string())

# top 10 rows of author_norm column in cleaned data
print("\nTop 10 rows of author_norm column in cleaned data:")
# cols: 'primary_author_name', 'author_2', 'author_3','author_4', 'author_5', 'author_1_role', 'author_2_role','author_3_role', 
# 'author_4_role', 'author_5_role'
# where author_1_role or author_2_role or author_3_role or author_4_role or author_5_role not null (so we can see some examples of author roles)
# how do i get not nulls?
print(cleaned_books_data[['primary_author_name', 'author_2', 'author_3',
         'author_4', 'author_5', 'author_1_role', 'author_2_role',
          'author_3_role', 'author_4_role', 'author_5_role']].head(10).to_string())




Top 10 rows of author column in uncleaned data:
0                                      Suzanne Collins
1            J.K. Rowling, Mary GrandPré (Illustrator)
2                                           Harper Lee
3            Jane Austen, Anna Quindlen (Introduction)
4                                      Stephenie Meyer
5                      Markus Zusak (Goodreads Author)
6    George Orwell, Russell Baker (Preface), C.M. W...
7             C.S. Lewis, Pauline Baynes (Illustrator)
8                                       J.R.R. Tolkien
9                                    Margaret Mitchell

Top 10 rows of author_norm column in cleaned data:
  primary_author_name             author_2 author_3 author_4 author_5 author_1_role author_2_role author_3_role author_4_role author_5_role
0       napoleon hill  ben holden-crowther     None     None     None          None          None          None          None          None
1             kj soze                 None     None     None     None

In [13]:
# books where has_multiple_ediitions is True
# has_multiple_editions is a column I created in the clean_data.py file 
# that is 1 if the book has more than 1 edition 
# this is calculated using book_uid and flags if edition_count > 1
print("\nBooks where has_multiple_editions is True:")
print(cleaned_books_data[cleaned_books_data['has_multiple_editions'] == True][['title', 'author_norm', 'edition', 'has_multiple_editions']].head(5).to_string())

# get all books where title is "Little Women"
print("\nBooks where title is 'Little Women':")
print(cleaned_books_data[cleaned_books_data['title'] == 'Little Women'].to_string())


Books where has_multiple_editions is True:
                        title                              author_norm edition  has_multiple_editions
66               Little Women                        louisa may alcott     NaN                      1
79            Vampire Academy                            richelle mead     NaN                      1
219  The Once and Future King                                 th white     NaN                      1
234            Norwegian Wood  haruki murakami, jay rubin (translator)     NaN                      1
268          The Way of Kings                        brandon sanderson     NaN                      1

Books where title is 'Little Women':
                     bookId         title                   series             author  rating                                                                                                                                                                                                                     

In [ ]:
# get rows where is_boxset is True
print("\nBooks where is_boxset is True:")
print(cleaned_books_data[cleaned_books_data['is_boxset'] == True][['title', 'title_norm', 'series', 'series_name_norm', 'author_norm', 'edition', 'is_boxset']].head(5).to_string())


Books where is_boxset is True:
                                                                     title                                                             title_norm                       series        series_name_norm  series_num                                                                                            author_norm                        edition  is_boxset
24   J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings  j.r.r. tolkien 4-book boxed set: the hobbit and the lord of the rings   The Lord of the Rings #0-3   the lord of the rings         NaN                                                                                            jrr tolkien  Hobbit Movie Tie-in Boxed set          1
77                                                  The Brothers Karamazov                                                 the brothers karamazov  The Brothers Karamazov #1-4  the brothers karamazov         NaN  fyodor dostoyevsky, fyodor dostoyevsky, rich

In [24]:
# check is_series is True
# we clean series from the original series column
# first we use parse_series() which exracts series name and number from the raw string
# is_part_of_series() checks if series_name_norm is not null
# create_series_identifier groups books into the same series by combining series_name_norm + primary_author_name
print("\nBooks where is_series is True:")
print(cleaned_books_data[cleaned_books_data['is_series'] == True][['title', 'title_norm', 'series', 'series_name_norm', 'series_num', 'author_norm', 'edition', 'is_series']].head(5).to_string()) 

                                                                   


Books where is_series is True:
                        title                 title_norm                series   series_name_norm  series_num      author_norm        edition  is_series
10      The Leopard Stratagem      the leopard stratagem  Leopard King Saga #2  leopard king saga         2.0          ta uner            NaN       True
15  The Muse of Edouard Manet  the muse of edouard manet     Edouard and Emily  edouard and emily         NaN       m clifford            NaN       True
17                      Nexus                      nexus     Supernova Saga #3     supernova saga         3.0        cl parker            NaN       True
18        Rome's Fallen Eagle        rome's fallen eagle          Vespasian #4          vespasian         4.0    robert fabbri            NaN       True
19           The Hunger Games           the hunger games   The Hunger Games #1   the hunger games         1.0  suzanne collins  First Edition       True
